In [ ]:
# ============================================================
# NOTEBOOK 02
# CT PREPROCESSING
# ============================================================

import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# KAGGLE INPUT PATHS
# ============================================================

CT_PATH = "/kaggle/input/datasets/waboke/myct-imgs/RibFrac1-image.nii/RibFrac1-image.nii"

LABEL_PATH = "/kaggle/input/datasets/waboke/myct-imgs/RibFrac1-label.nii/RibFrac1-label.nii"

print("CT:", CT_PATH)
print("Label:", LABEL_PATH)

In [ ]:
# ============================================================
# LOAD NIFTI DATA
# ============================================================

ct_img = nib.load(CT_PATH)
label_img = nib.load(LABEL_PATH)

ct_volume = ct_img.get_fdata().astype(np.float32)
label_volume = label_img.get_fdata().astype(np.int16)

print("CT loaded successfully.")
print("Label loaded successfully.")

print("CT shape:", ct_volume.shape)
print("Label shape:", label_volume.shape)

In [ ]:
# ============================================================
# DATA VALIDATION
# ============================================================

assert ct_volume.ndim == 3
assert label_volume.ndim == 3

assert ct_volume.shape == label_volume.shape

ct_spacing = ct_img.header.get_zooms()[:3]
label_spacing = label_img.header.get_zooms()[:3]

print("CT shape:", ct_volume.shape)
print("Label shape:", label_volume.shape)

print("CT spacing:", ct_spacing)
print("Label spacing:", label_spacing)

In [ ]:
# ============================================================
# CONVERT CT TO FLOAT32
# ============================================================

ct_volume = np.asarray(ct_volume, dtype=np.float32)

print("CT data type:", ct_volume.dtype)

In [ ]:
# ============================================================
# ORIGINAL CT INTENSITY STATISTICS
# ============================================================

print("Original CT statistics")
print("--------------------------------")
print("Minimum:", np.min(ct_volume))
print("Maximum:", np.max(ct_volume))
print("Mean:", np.mean(ct_volume))
print("Standard deviation:", np.std(ct_volume))

In [ ]:
# ============================================================
# BONE WINDOW PARAMETERS
# ============================================================

WINDOW_LEVEL = 500
WINDOW_WIDTH = 2000

WINDOW_MIN = WINDOW_LEVEL - (WINDOW_WIDTH / 2)
WINDOW_MAX = WINDOW_LEVEL + (WINDOW_WIDTH / 2)

print("Bone window:")
print("Window level:", WINDOW_LEVEL)
print("Window width:", WINDOW_WIDTH)
print("Lower HU:", WINDOW_MIN)
print("Upper HU:", WINDOW_MAX)

In [ ]:
# ============================================================
# APPLY BONE-WINDOW CLIPPING
# ============================================================

ct_windowed = np.clip(
    ct_volume,
    WINDOW_MIN,
    WINDOW_MAX
)

print("Windowed CT statistics")
print("--------------------------------")
print("Minimum:", np.min(ct_windowed))
print("Maximum:", np.max(ct_windowed))

In [ ]:
# ============================================================
# MIN-MAX NORMALIZATION
# ============================================================

ct_normalized = ( ct_windowed - WINDOW_MIN) / ( WINDOW_MAX - WINDOW_MIN )

ct_normalized = ct_normalized.astype(np.float32)

print("Normalized CT statistics")
print("--------------------------------")
print("Minimum:", np.min(ct_normalized))
print("Maximum:", np.max(ct_normalized))
print("Mean:", np.mean(ct_normalized))
print("Standard deviation:", np.std(ct_normalized))
print("Data type:", ct_normalized.dtype)

In [ ]:
# ============================================================
# VERIFY LABEL INTEGRITY
# ============================================================

unique_labels = np.unique(label_volume)

print("Unique label values:")
print(unique_labels)

In [ ]:
# ============================================================
# CHECK CT AND LABEL ALIGNMENT
# ============================================================

assert ct_normalized.shape == label_volume.shape

print("CT shape:", ct_normalized.shape)
print("Label shape:", label_volume.shape)

print("Spatial dimensions remain aligned.")

In [ ]:
# ============================================================
# ORIGINAL CT
# ============================================================

middle_slice = ct_volume.shape[2] // 2

plt.figure(figsize=(7, 7))

plt.imshow(
    ct_volume[:, :, middle_slice],
    cmap="gray"
)

plt.title(f"Original CT - Slice {middle_slice}")
plt.axis("off")

plt.show()

In [ ]:
# ============================================================
# BONE-WINDOWED CT
# ============================================================

plt.figure(figsize=(7, 7))

plt.imshow(
    ct_windowed[:, :, middle_slice],
    cmap="gray",
    vmin=WINDOW_MIN,
    vmax=WINDOW_MAX
)

plt.title(
    f"Bone-Windowed CT - Slice {middle_slice}"
)

plt.axis("off")

plt.show()

In [ ]:
# ============================================================
# NORMALIZED CT
# ============================================================

plt.figure(figsize=(7, 7))

plt.imshow(
    ct_normalized[:, :, middle_slice],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title(
    f"Normalized CT - Slice {middle_slice}"
)

plt.axis("off")

plt.show()

In [ ]:
# ============================================================
# FRACTURE ANNOTATION
# ============================================================

plt.figure(figsize=(7, 7))

plt.imshow(
    label_volume[:, :, middle_slice]
)

plt.title(
    f"Fracture Annotation - Slice {middle_slice}"
)

plt.axis("off")

plt.show()

In [ ]:
# ============================================================
# NORMALIZED CT + FRACTURE ANNOTATION
# ============================================================

plt.figure(figsize=(8, 8))

plt.imshow(
    ct_normalized[:, :, middle_slice],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.imshow(
    label_volume[:, :, middle_slice],
    alpha=0.4
)

plt.title(
    f"Preprocessed CT + Fracture Annotation\n"
    f"Slice {middle_slice}"
)

plt.axis("off")

plt.show()

In [ ]:
# ============================================================
# FINAL PREPROCESSING VALIDATION
# ============================================================

assert ct_normalized.dtype == np.float32

assert np.isfinite(ct_normalized).all()

assert np.min(ct_normalized) >= 0.0

assert np.max(ct_normalized) <= 1.0

assert ct_normalized.shape == label_volume.shape

assert np.isfinite(label_volume).all()

print("==============================================")
print("NOTEBOOK 02 - PREPROCESSING SUCCESSFUL")
print("==============================================")

print("Original CT shape:", ct_volume.shape)
print("Processed CT shape:", ct_normalized.shape)

print("Original CT dtype:", ct_volume.dtype)
print("Processed CT dtype:", ct_normalized.dtype)

print("Window level:", WINDOW_LEVEL)
print("Window width:", WINDOW_WIDTH)

print("Normalized minimum:", np.min(ct_normalized))
print("Normalized maximum:", np.max(ct_normalized))

print("Label values:", np.unique(label_volume))

print("==============================================")

In [ ]:
# ============================================================
# SAVE PREPROCESSED DATA
# ============================================================

OUTPUT_DIR = "/kaggle/working/preprocessed_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

ct_output_path = os.path.join(
    OUTPUT_DIR,
    "RibFrac1_ct_normalized.npy"
)

label_output_path = os.path.join(
    OUTPUT_DIR,
    "RibFrac1_label.npy"
)

np.save(ct_output_path, ct_normalized)
np.save(label_output_path, label_volume)

print("Preprocessed data saved.")
print()
print("CT:", ct_output_path)
print("Label:", label_output_path)

In [ ]:
# ============================================================
# SAVE PREPROCESSING PARAMETERS
# ============================================================

metadata = {
    "ct_original_shape": list(ct_volume.shape),
    "ct_spacing": list(ct_spacing),
    "label_shape": list(label_volume.shape),
    "window_level": WINDOW_LEVEL,
    "window_width": WINDOW_WIDTH,
    "window_min": WINDOW_MIN,
    "window_max": WINDOW_MAX,
    "normalization": "min-max to [0, 1]",
    "ct_dtype": str(ct_normalized.dtype),
    "label_dtype": str(label_volume.dtype)
}

metadata_path = os.path.join(
    OUTPUT_DIR,
    "preprocessing_metadata.npy"
)

np.save(
    metadata_path,
    metadata,
    allow_pickle=True
)

print("Metadata saved to:")
print(metadata_path)

In [ ]:
# ============================================================
# VERIFY SAVED OUTPUTS
# ============================================================

print("Saved files:")

for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        file_path = os.path.join(root, file)
        file_size_mb = os.path.getsize(file_path) / (1024 ** 2)

        print(
            f"{file}: {file_size_mb:.2f} MB"
        )